# 04 — Model Training and Evaluation

**Team:** StellarX  
**Phase:** 3 — Neural Network / Pattern Recognition  
**Status:** ✅ Fully executable — Phase 3 sklearn backend active.

## What this notebook covers

1. Load feature dataset and trained checkpoint
2. Classifier comparison (RandomForest vs KNN vs MLP)
3. Confusion matrix analysis
4. Feature importance (RandomForest)
5. Confidence calibration
6. Inference latency benchmarking
7. Results summary table
8. Phase 4 readiness checklist

In [ ]:
import warnings, math, time
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import yaml
from pathlib import Path

warnings.filterwarnings('ignore')
matplotlib.use('Agg')
%matplotlib inline

with open('config.yaml') as f:
    config = yaml.safe_load(f)

print('Configuration loaded.')

---
## 1. Load feature dataset

In [ ]:
from src.preprocessing.feature_dataset import build_feature_dataset, load_feature_dataset

DATASET_DIR = Path('data/processed/features')

if (DATASET_DIR / 'X.npy').exists():
    X_all, y_all, meta_all = load_feature_dataset(DATASET_DIR)
    print(f'Loaded cached dataset: {X_all.shape}')
else:
    print('Building feature dataset…')
    X_all, y_all, meta_all = build_feature_dataset(config, verbose=True, save_path=DATASET_DIR)

splits = np.array([m['split'] for m in meta_all])
X_train, y_train = X_all[splits=='train'], y_all[splits=='train']
X_val,   y_val   = X_all[splits=='val'],   y_all[splits=='val']
X_test,  y_test  = X_all[splits=='test'],  y_all[splits=='test']

print(f'Train: {len(X_train)}  Val: {len(X_val)}  Test: {len(X_test)}')
print(f'Unique labels — Train: {len(np.unique(y_train))}  '
      f'Val: {len(np.unique(y_val))}  Test: {len(np.unique(y_test))}')

---
## 2. Classifier comparison

In [ ]:
from src.models.sklearn_classifier import StarPatternClassifier, evaluate_classifier

classifier_types = ['random_forest', 'knn', 'mlp']
comparison_rows = []

for ctype in classifier_types:
    cfg_c = {**config, 'model': {**config['model'], 'classifier_type': ctype}}
    clf_c = StarPatternClassifier(cfg_c)
    t0 = time.time()
    result = clf_c.fit(X_train, y_train, seed=42)
    train_time = time.time() - t0

    metrics_val = evaluate_classifier(clf_c, X_val, y_val, config) if len(X_val) else {}
    metrics_test = evaluate_classifier(clf_c, X_test, y_test, config) if len(X_test) else {}

    # Inference latency (single sample, mean of 50 runs)
    if len(X_test) > 0:
        t_inf = []
        for _ in range(50):
            t0 = time.perf_counter()
            clf_c.predict(X_test[0])
            t_inf.append((time.perf_counter() - t0) * 1000)
        lat_ms = np.mean(t_inf)
    else:
        lat_ms = float('nan')

    comparison_rows.append({
        'Classifier':    ctype,
        'Train acc':     round(result.train_accuracy, 4),
        'Val top-1':     round(metrics_val.get('top1_accuracy', float('nan')), 4),
        'Test top-1':    round(metrics_test.get('top1_accuracy', float('nan')), 4),
        f'Test top-{config["evaluation"]["top_k"]}':  round(metrics_test.get('topk_accuracy', float('nan')), 4),
        'Train time (s)': round(train_time, 2),
        'Latency (ms)':   round(lat_ms, 3),
    })
    print(f'  {ctype:15s}: train_acc={result.train_accuracy:.4f}  '
          f'val={metrics_val.get("top1_accuracy", float("nan")):.4f}  '
          f'lat={lat_ms:.2f}ms')

df_cmp = pd.DataFrame(comparison_rows)
print()
print(df_cmp.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(comparison_rows))
w = 0.25
ax.bar(x - w, df_cmp['Train acc'],  w, label='Train', color='steelblue')
ax.bar(x,     df_cmp['Val top-1'],  w, label='Val',   color='mediumseagreen')
ax.bar(x + w, df_cmp['Test top-1'], w, label='Test',  color='salmon')
ax.set_xticks(x)
ax.set_xticklabels(df_cmp['Classifier'])
ax.set_ylim(0, 1.05)
ax.set_ylabel('Top-1 Accuracy')
ax.set_title('Classifier Comparison — Top-1 Accuracy')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('notebooks/fig_13_classifier_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure saved → notebooks/fig_13_classifier_comparison.png')

---
## 3. Confusion matrix (RandomForest)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from src.models.sklearn_classifier import train_classifier

# Re-train the RF (use the default config)
clf_rf, _ = train_classifier(X_train, y_train, config, seed=42)

eval_X = X_test if len(X_test) >= 4 else X_val if len(X_val) >= 4 else X_train
eval_y = y_test if len(X_test) >= 4 else y_val  if len(X_val) >= 4 else y_train

labels_pred, _ = clf_rf.predict_batch(eval_X)
unique_classes = sorted(np.unique(np.concatenate([eval_y, labels_pred])))

if len(unique_classes) <= 30:
    cm = confusion_matrix(eval_y, labels_pred, labels=unique_classes)
    fig, ax = plt.subplots(figsize=(max(6, len(unique_classes)*0.5+2),
                                    max(5, len(unique_classes)*0.5+1.5)))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                   display_labels=[f'c{l}' for l in unique_classes])
    disp.plot(ax=ax, colorbar=False, xticks_rotation='vertical')
    ax.set_title(f'Confusion Matrix — RandomForest ({len(eval_X)} samples)')
    plt.tight_layout()
    plt.savefig('notebooks/fig_14_confusion_matrix.png', dpi=120, bbox_inches='tight')
    plt.show()
    n_correct = int(np.trace(cm))
    print(f'Correctly classified: {n_correct}/{len(eval_X)}  ({n_correct/len(eval_X)*100:.1f}%)')
    print('Figure saved → notebooks/fig_14_confusion_matrix.png')
else:
    print(f'Too many classes ({len(unique_classes)}) for readable confusion matrix — skipping plot.')
    correct = (labels_pred == eval_y).sum()
    print(f'Accuracy: {correct}/{len(eval_y)} = {correct/len(eval_y):.4f}')

---
## 4. Feature importance (RandomForest)

In [ ]:
N_PAIRS = config['features']['max_stars'] * (config['features']['max_stars'] - 1) // 2

if hasattr(clf_rf._clf, 'feature_importances_'):
    importances = clf_rf._clf.feature_importances_
    feat_labels = ([f'd_{i}' for i in range(N_PAIRS)] +
                   [f'r_{i}' for i in range(N_PAIRS)])

    top_n = 20
    idx = np.argsort(importances)[::-1][:top_n]

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.bar(range(top_n), importances[idx], color='steelblue', edgecolor='white')
    ax.set_xticks(range(top_n))
    ax.set_xticklabels([feat_labels[i] for i in idx], rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('Importance')
    ax.set_title(f'Top-{top_n} Feature Importances (RandomForest)\n'
                 f'd_* = pairwise distance, r_* = brightness ratio')
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig('notebooks/fig_15_feature_importance.png', dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Sum of top-{top_n} importances: {importances[idx].sum():.4f}')
    print('Figure saved → notebooks/fig_15_feature_importance.png')

---
## 5. Confidence calibration

In [ ]:
eval_X = X_test if len(X_test) >= 4 else X_train
eval_y = y_test if len(X_test) >= 4 else y_train
labels_pred, confs = clf_rf.predict_batch(eval_X)
correct = (labels_pred == eval_y).astype(float)

# Reliability diagram: bucket confidences, compute accuracy per bucket
n_bins = 10
bins = np.linspace(0, 1, n_bins + 1)
bucket_acc, bucket_conf, bucket_n = [], [], []
for lo, hi in zip(bins[:-1], bins[1:]):
    mask = (confs >= lo) & (confs < hi)
    if mask.sum() > 0:
        bucket_acc.append(correct[mask].mean())
        bucket_conf.append(confs[mask].mean())
        bucket_n.append(mask.sum())

bucket_acc  = np.array(bucket_acc)
bucket_conf = np.array(bucket_conf)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Perfect calibration')
ax.scatter(bucket_conf, bucket_acc, s=60, color='steelblue', zorder=3)
ax.plot(bucket_conf, bucket_acc, '-', color='steelblue', label='Classifier')
ax.set_xlabel('Mean predicted confidence')
ax.set_ylabel('Fraction correct')
ax.set_title('Reliability Diagram (Confidence Calibration)')
ax.set_xlim(0, 1); ax.set_ylim(0, 1.05)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.hist(confs[correct == 1], bins=20, alpha=0.7, color='seagreen',  label='Correct')
ax.hist(confs[correct == 0], bins=20, alpha=0.7, color='tomato',   label='Incorrect')
ax.axvline(config['evaluation']['confidence_threshold'], color='gray', linestyle='--',
           label='Threshold')
ax.set_xlabel('Confidence')
ax.set_ylabel('Frames')
ax.set_title('Confidence Distribution — Correct vs Incorrect')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('notebooks/fig_16_confidence_calibration.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure saved → notebooks/fig_16_confidence_calibration.png')

---
## 6. Inference latency benchmark

In [ ]:
from src.models.inference import run_inference

n_bench = min(200, len(eval_X))
latencies_ms = []
for i in range(n_bench):
    result = run_inference(clf_rf, eval_X[i], config)
    latencies_ms.append(result.latency_ms)

lat = np.array(latencies_ms)
print(f'Inference latency over {n_bench} samples:')
print(f'  Mean   : {lat.mean():.3f} ms')
print(f'  Median : {np.median(lat):.3f} ms')
print(f'  P95    : {np.percentile(lat, 95):.3f} ms')
print(f'  P99    : {np.percentile(lat, 99):.3f} ms')
print(f'  Max    : {lat.max():.3f} ms')

fig, ax = plt.subplots(figsize=(9, 3))
ax.hist(lat, bins=30, color='cornflowerblue', edgecolor='white')
ax.axvline(lat.mean(), color='tomato', linestyle='--',
           label=f'Mean = {lat.mean():.2f} ms')
ax.axvline(np.percentile(lat, 95), color='orange', linestyle=':',
           label=f'P95 = {np.percentile(lat,95):.2f} ms')
ax.set_xlabel('Latency (ms)')
ax.set_ylabel('Samples')
ax.set_title('Inference Latency Distribution (single sample, sklearn RF)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('notebooks/fig_17_inference_latency.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure saved → notebooks/fig_17_inference_latency.png')

---
## 7. Results summary

In [ ]:
_, confs_test = clf_rf.predict_batch(X_test) if len(X_test) else (None, np.array([]))
labels_test, _ = clf_rf.predict_batch(X_test) if len(X_test) else (np.array([]), None)
test_acc = (labels_test == y_test).mean() if len(X_test) else float('nan')

print('=' * 55)
print('Phase 3 Results Summary')
print('=' * 55)
print(f'  Classifier        : {config["model"]["classifier_type"]}')
print(f'  Feature dim       : {X_train.shape[1]}')
print(f'  Training samples  : {len(X_train)}')
print(f'  Unique classes    : {len(np.unique(y_train))} / {config["model"].get("n_sky_cells",500)}')
print(f'  Test top-1 acc    : {test_acc:.4f}')
print(f'  Mean infer latency: {lat.mean():.3f} ms')
print(f'  Checkpoint        : models/{config["model"]["checkpoint_name"]}')
print('=' * 55)
print()
print('Note: Low accuracy reflects the sparse 50-star prototype catalog.')
print('Many frames yield 0 detected stars → zero feature vectors → random predictions.')
print('Accuracy will improve substantially once the full Hipparcos catalog is loaded.')

---
## 8. Phase 4 readiness checklist

| Item | Status |
|---|---|
| Feature extraction pipeline | ✅ Implemented and tested |
| Classifier training + evaluation | ✅ RF / KNN / MLP all working |
| Checkpoint save/load | ✅ Verified round-trip |
| `run_inference()` → `RecognitionResult` | ✅ Confidence-gated, top-k |
| Extend catalog to full Hipparcos | ⬜ Needed before Phase 4 |
| Implement `pattern_matcher.py` | ⬜ Phase 4 |
| Match sky-cell ID → catalog star IDs | ⬜ Phase 4 |
| Attitude estimation from matches | ⬜ Phase 5 |